# DATA CLEANING & EXPLORATORY DATA ANALYSIS

**Input:** `../collected_data/combined_jobs.csv`
**Output:** `../data/processed/combined_jobs_cleaned.csv`

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

ROOT        = Path('..').resolve()
INPUT_PATH  = ROOT / 'collected_data' / 'combined_jobs.csv'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'combined_jobs_cleaned.csv'
PLOTS_PATH  = ROOT / 'outputs' / 'plots'

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
PLOTS_PATH.mkdir(parents=True, exist_ok=True)

In [20]:
# LOAD
df = pd.read_csv(INPUT_PATH)
print(f"Raw shape: {df.shape}")

Raw shape: (7321, 18)


In [21]:
# MISSING VALUE REPORT
print("\n--- Missing Values (Raw) ---")
miss = pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Missing %': (df.isnull().mean() * 100).round(1)
})
print(miss[miss['Missing'] > 0].to_string())


--- Missing Values (Raw) ---
                 Missing  Missing %
job_id               209        2.9
company_name         727        9.9
location             678        9.3
salary_min          1241       17.0
salary_max          1247       17.0
job_description      737       10.1
contract_type       4423       60.4
contract_time       4092       55.9
posted_date          737       10.1
job_url               64        0.9
category            1237       16.9
search_keyword      7257       99.1
search_location     7257       99.1
access_type         6821       93.2


In [22]:
# DROP NEAR-USELESS COLUMNS (>99% missing)
drop_cols = [c for c in ['search_keyword', 'search_location', 'access_type'] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print(f"\nDropped columns: {drop_cols}")


Dropped columns: ['search_keyword', 'search_location', 'access_type']


In [23]:
# REMOVE DUPLICATES
before = len(df)
df.drop_duplicates(
    subset=['job_title', 'company_name', 'location'], keep='first', inplace=True
)
print(f"Duplicates removed : {before - len(df)}  |  Remaining: {len(df)}")

Duplicates removed : 4522  |  Remaining: 2799


In [24]:
# DROP ROWS WITH NO JOB DESCRIPTION
before = len(df)
df = df[df['job_description'].notna() & (df['job_description'].str.strip() != '')]
print(f"No-description rows dropped : {before - len(df)}  |  Remaining: {len(df)}")

No-description rows dropped : 77  |  Remaining: 2722


In [25]:
# FIX INDIVIDUAL COLUMNS

# company_name & location
df['company_name'] = df['company_name'].fillna('Unknown').str.strip()
df['location']     = df['location'].fillna('Unknown').str.strip()

# salary - replace 0 and negatives with NaN
df['salary_min'] = df['salary_min'].replace(0, np.nan)
df['salary_min'] = df['salary_min'].apply(lambda x: np.nan if pd.notna(x) and x < 0 else x)
df['salary_max'] = df['salary_max'].apply(lambda x: np.nan if pd.notna(x) and x < 100 else x)
df['salary_avg'] = np.where(
    df['salary_min'].notna() & df['salary_max'].notna(),
    (df['salary_min'] + df['salary_max']) / 2, np.nan
)

# contract
df['contract_type'] = df['contract_type'].fillna('Unknown')
df['contract_time'] = df['contract_time'].fillna('Unknown')

# category
df['category'] = df['category'].fillna('Unknown')

# dates
df['posted_date']    = pd.to_datetime(df['posted_date'],    errors='coerce', utc=True)
df['collected_date'] = pd.to_datetime(df['collected_date'], errors='coerce', utc=True)
df['days_since_posted'] = (df['collected_date'] - df['posted_date']).dt.days
df['posted_month']      = df['posted_date'].dt.to_period('M').astype(str)
df['posted_year']       = df['posted_date'].dt.year

In [26]:
# TEXT CLEANING
def clean_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = re.sub(r'<[^>]+>', ' ', str(text))    # strip HTML
    text = re.sub(r'http\S+', '', text)            # strip URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)   # alphanumeric only
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['job_description_clean'] = df['job_description'].apply(clean_text)

# Derived text features
df['desc_length']     = df['job_description'].str.len()
df['desc_word_count'] = df['job_description_clean'].str.split().str.len()
df['is_truncated']    = df['job_description'].str.strip().str.endswith('\u2026').astype(int)
df['title_length']    = df['job_title'].str.len()
df['title_word_count']= df['job_title'].str.split().str.len()

# Has salary in description text (useful signal)
df['salary_in_desc']  = df['job_description_clean'].str.contains(
    r'\b(salary|£|gbp|per annum|per hour|hourly|annual)\b', regex=True
).astype(int)

# Has contact info in description (red flag)
df['contact_in_desc'] = df['job_description_clean'].str.contains(
    r'\b(whatsapp|telegram|gmail|yahoo|hotmail|call us|text us)\b', regex=True
).astype(int)

In [27]:
# FINAL SUMMARY
print("\n--- Final Clean Dataset ---")
print(f"Total rows    : {len(df)}")
print(f"Total columns : {len(df.columns)}")
print(f"\nPlatform distribution:")
print(df['source_platform'].value_counts().to_string())
print(f"\nTop categories:")
print(df['category'].value_counts().head(8).to_string())
print(f"\nSalary avg — median : £{df['salary_avg'].median():,.0f}")
print(f"Salary avg — 95th % : £{df['salary_avg'].quantile(0.95):,.0f}")
print(f"Truncated desc      : {df['is_truncated'].sum()} ({df['is_truncated'].mean()*100:.1f}%)")
print(f"Missing salary_avg  : {df['salary_avg'].isna().sum()} ({df['salary_avg'].isna().mean()*100:.1f}%)")
print(f"Missing company     : {(df['company_name']=='Unknown').sum()}")
print(f"Salary in desc      : {df['salary_in_desc'].sum()}")
print(f"Contact info in desc: {df['contact_in_desc'].sum()}")

df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved → {OUTPUT_PATH}")


--- Final Clean Dataset ---
Total rows    : 2722
Total columns : 27

Platform distribution:
source_platform
Adzuna     2519
TheMuse     203

Top categories:
category
IT Jobs                             2078
Unknown                              203
Engineering Jobs                     183
Trade & Construction Jobs             85
Accounting & Finance Jobs             33
PR, Advertising & Marketing Jobs      29
Scientific & QA Jobs                  21
Sales Jobs                            15

Salary avg — median : £57,592
Salary avg — 95th % : £119,522
Truncated desc      : 2504 (92.0%)
Missing salary_avg  : 276 (10.1%)
Missing company     : 20
Salary in desc      : 612
Contact info in desc: 8

Saved → C:\Users\Nethma\Desktop\Assigment\18k research\webscrape\AI_trust\2_development\data\processed\combined_jobs_cleaned.csv


In [28]:
# EDA PLOT 1 - Overview
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA Overview — Job Postings Dataset\n'
             'Trust & Transparency in AI-Powered Recruitment',
             fontsize=13, fontweight='bold')

# Plot 1: Jobs by Platform
ax = axes[0, 0]
pc = df['source_platform'].value_counts()
bars = ax.bar(pc.index, pc.values, color=sns.color_palette("muted", len(pc)))
ax.set_title('Jobs by Source Platform', fontweight='bold')
ax.set_ylabel('Count')
for b, v in zip(bars, pc.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height()+8, str(v), ha='center', fontsize=9)

# Plot 2: Top Categories
ax = axes[0, 1]
tc = df['category'].value_counts().head(10)
ax.barh(tc.index[::-1], tc.values[::-1], color=sns.color_palette("muted", 10))
ax.set_title('Top 10 Job Categories', fontweight='bold')
ax.set_xlabel('Count')

# Plot 3: Salary Distribution
ax = axes[0, 2]
sal = df['salary_avg'].dropna()
sal = sal[sal < 200000]
ax.hist(sal, bins=40, color='steelblue', edgecolor='white')
ax.axvline(sal.median(), color='red', linestyle='--', label=f'Median £{sal.median():,.0f}')
ax.axvline(sal.quantile(0.95), color='orange', linestyle='--',
           label=f'95th £{sal.quantile(0.95):,.0f}')
ax.set_title('Salary Distribution (GBP)', fontweight='bold')
ax.set_xlabel('Average Salary')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.legend(fontsize=8)

# Plot 4: Description Word Count
ax = axes[1, 0]
ax.hist(df['desc_word_count'].dropna().clip(upper=500), bins=40,
        color='mediumseagreen', edgecolor='white')
ax.axvline(50, color='red', linestyle='--', label='Short (<50 words)')
ax.set_title('Description Word Count', fontweight='bold')
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend(fontsize=8)

# Plot 5: Missing data %
ax = axes[1, 1]
key_cols  = ['company_name', 'salary_min', 'salary_max',
             'contract_type', 'contract_time', 'category', 'location']
miss_vals = []
for c in key_cols:
    if df[c].dtype == object:
        miss_vals.append((df[c] == 'Unknown').mean() * 100)
    else:
        miss_vals.append(df[c].isna().mean() * 100)
colors_m = ['#F44336' if p > 40 else '#FF9800' if p > 15 else '#4CAF50'
            for p in miss_vals]
bars = ax.barh(key_cols, miss_vals, color=colors_m)
ax.set_title('Missing / Unknown Data %', fontweight='bold')
ax.set_xlabel('% Missing')
for b, v in zip(bars, miss_vals):
    ax.text(v+0.5, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=8)

# Plot 6: Contract type
ax = axes[1, 2]
known = df[df['contract_type'] != 'Unknown']['contract_type'].value_counts()
ax.pie(known.values, labels=known.index, autopct='%1.1f%%',
       colors=sns.color_palette("muted", len(known)), startangle=90)
ax.set_title('Contract Type (Known Only)', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '1_eda_overview.png', bbox_inches='tight')
plt.close()
print("Saved → 1_eda_overview.png")

Saved → 1_eda_overview.png


In [29]:
# EDA PLOT 2 - Distributions deep dive
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('EDA — Text & Time Distributions', fontsize=13, fontweight='bold')

# Title word count
ax = axes[0]
ax.hist(df['title_word_count'].dropna(), bins=20, color='mediumpurple', edgecolor='white')
ax.set_title('Job Title Word Count', fontweight='bold')
ax.set_xlabel('Words')
ax.set_ylabel('Frequency')

# Salary by platform
ax = axes[1]
platforms_with_salary = df[df['salary_avg'].notna()]
platform_order = platforms_with_salary['source_platform'].value_counts().index.tolist()
sal_data = [platforms_with_salary[platforms_with_salary['source_platform'] == p]['salary_avg'].clip(upper=200000)
            for p in platform_order]
ax.boxplot(sal_data, labels=platform_order, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_title('Salary Distribution by Platform', fontweight='bold')
ax.set_ylabel('Salary (GBP)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))

# Postings over time
ax = axes[2]
time_counts = df.groupby('posted_month').size().reset_index(name='count')
time_counts = time_counts[time_counts['posted_month'] != 'NaT'].tail(12)
ax.bar(range(len(time_counts)), time_counts['count'], color='coral', edgecolor='white')
ax.set_xticks(range(len(time_counts)))
ax.set_xticklabels(time_counts['posted_month'], rotation=45, ha='right', fontsize=7)
ax.set_title('Job Postings Over Time', fontweight='bold')
ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '1_eda_distributions.png', bbox_inches='tight')
plt.close()
print("Saved → 1_eda_distributions.png")

Saved → 1_eda_distributions.png
